# Attention Sink Analysis

Investigate attention sinks (positions that receive disproportionate attention): sink position identification, sink head detection, magnitude tracking, impact of sink removal, and sink formation analysis.

## Why This Matters

Attention sink reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_sink_analysis import (
    sink_position_identification,
    sink_head_detection,
    sink_magnitude_tracking,
    sink_removal_impact,
    sink_formation_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
# Identify sink positions
result = sink_position_identification(model, tokens, threshold=0.15)
print(f'Sink count: {result["sink_count"]}')
print(f'Strongest sink: position {result["strongest_sink"]}')
print('Per-position scores:', np.array(result['per_position_scores']).round(3))

In [ ]:
# Detect sink heads
result = sink_head_detection(model, tokens, sink_pos=0, threshold=0.15)
print(f'Sink head fraction: {result["sink_head_fraction"]:.3f}')
for h in result['sink_heads'][:5]:
    print(f'  L{h["layer"]}H{h["head"]}: mean={h["mean_attention_to_sink"]:.4f}')

In [ ]:
# Track sink magnitude across layers
result = sink_magnitude_tracking(model, tokens)
print(f'Peak layer: {result["peak_layer"]}')
print(f'Trend: {result["trend"]:.4f}')
for layer in result['per_layer']:
    print(f'  Layer {layer["layer"]}: mean={layer["mean_attention"]:.4f}, '
          f'max={layer["max_attention"]:.4f}')

In [ ]:
# Impact of removing attention to the sink
result = sink_removal_impact(model, tokens, sink_pos=0, layer=0)
print(f'Output change norm: {result["output_change_norm"]:.4f}')
print(f'Logit change (last pos): {result["logit_change_last_pos"]:.4f}')

In [ ]:
# Analyze why positions become sinks
result = sink_formation_analysis(model, tokens)
for layer in result['per_layer']:
    print(f'Layer {layer["layer"]}: key_norm_ratio={layer["key_norm_ratio"]:.3f}, '
          f'value_norm_ratio={layer["value_norm_ratio"]:.3f}, '
          f'cross_head_sim={layer["cross_head_key_similarity"]:.3f}')